# Imbalanced Feature Screener on SBA loans

This notebook demonstrates the `ImbalancedFeatureScreener` package on the SBA imbalanced loan dataset.

The workflow: load the cleaned imbalanced dataset, fit the screener, inspect the ranked report, then visualize the summary and a representative feature diagnostic.

In [1]:
import sys
from pathlib import Path
import pandas as pd
sys.path.insert(0, '/home/rajiv/programming/imbalanced_feature_screener')
from feature_screener import ImbalancedFeatureScreener

data_path = Path('/home/rajiv/programming/imbalanced_feature_screener/data/sba_loans_imb_learn.csv')
df = pd.read_csv(data_path, low_memory=False)
print(df.shape)
print(df['target'].value_counts().sort_index().to_string())
X = df.drop(columns=['target'])
y = df['target']
print(f'Features: {len(X.columns)}')


/home/rajiv/programming/imbalanced_feature_screener/imb_feature_screener/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(75073, 38)
target
0    68180
1     6893
Features: 37


In [2]:
screener = ImbalancedFeatureScreener(top_k=10, n_splits=3)
screener.fit(X, y)
print(screener.report_.head(10)[['feature','iv','ks_stat','uni_pr_auc','perm_pr_auc','pos_shap','consensus_rank']].to_string(index=False))
print(f'\nSelected features: {screener.selected_features_}')


              feature       iv  ks_stat  uni_pr_auc  perm_pr_auc  pos_shap  consensus_rank
  initialinterestrate 0.172293 0.303055    0.197273 3.700743e-17  0.043315             3.8
         approvaldate 0.054645 0.205396    0.110829 3.700743e-17  0.021563             6.2
firstdisbursementdate 0.051592 0.197018    0.110026 7.401487e-17  0.000024             9.0
         terminmonths 0.425183 0.503512    0.060475 3.700743e-17  0.134882             9.4
           loanstatus 0.000000 1.000000    0.091817 8.542452e-01  1.615488            10.3
             bankcity 0.071845 0.197419    0.128404 3.700743e-17  0.000000            11.2
      loan_age_months 0.049182 0.197018    0.077401 3.700743e-17  0.003354            13.1
              bankzip 0.036044 0.132476    0.079466 7.401487e-17  0.004850            13.5
             bankname 0.013399 0.161202    0.121899 0.000000e+00  0.001933            13.6
    estimated_balance 0.190434 0.309522    0.062505 0.000000e+00  0.006282            14.0

In [3]:
X_selected = screener.transform(X)
print(X_selected.shape)
print(X_selected.head())


(75073, 10)
   initialinterestrate approvaldate firstdisbursementdate  terminmonths  \
0                11.04   2019-10-01            2019-10-31         120.0   
1                 9.00   2019-10-01            2019-10-31         120.0   
2                 6.99   2019-10-01            2020-02-29          60.0   
3                 6.00   2019-10-01            2019-12-31          90.0   
4                 7.00   2019-10-01            2020-01-31         120.0   

  loanstatus    bankcity  loan_age_months  bankzip  \
0      P I F  WILMINGTON        79.967146    19808   
1      P I F  WILMINGTON        79.967146    19801   
2      P I F  Cincinnati        75.991786    45202   
3      P I F     HOUSTON        77.963039    77036   
4      P I F  CINCINNATI        76.944559    45263   

                               bankname  estimated_balance  
0         TD Bank, National Association       11507.764607  
1  Santander Bank, National Association       41832.672735  
2       U.S. Bank, National A

In [4]:
summary_fig = screener.plot_summary(top_n=8)
summary_fig


<Figure size 1000x800 with 3 Axes>

In [5]:
feature = 'initialinterestrate'
feature_fig = screener.plot_feature(X, y, feature=feature, log_x=False)
feature_fig


<Figure size 1200x400 with 2 Axes>